# 0. 安装与导入


In [3]:
from typing import TypedDict, Literal

from langgraph.graph import StateGraph, START, END

# 1. State：Agent 流程里传递的数据
你要理解

State 就是整个 Agent 的共享工作台。

每个节点都能读取它，也可以返回一个字典来更新它。、

In [4]:
class AgentState(TypedDict):
    query: str
    task_type: str
    result: str
    final_answer: str

In [16]:
state = {
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "",
    "result": "",
    "final_answer": "",
}

state

{'query': '请帮我分析 coco_val_n300_g1 的幻觉风险',
 'task_type': '',
 'result': '',
 'final_answer': ''}

## LangGraph 不是让节点互相直接传参，而是让所有节点围绕同一个 State 工作。

# 2. Node：节点就是处理 State 的函数
   


2.1 分类节点 classify_task

这个节点负责根据用户问题判断任务类型

In [17]:
def classify_task(state: AgentState) -> dict:
    query = state["query"]

    if "论文" in query or "paper" in query.lower():
        task_type = "paper_question"
    elif "实验" in query or "coco" in query.lower() or "幻觉" in query:
        task_type = "experiment_analysis"
    elif "数据集" in query or "dataset" in query.lower():
        task_type = "dataset_recommendation"
    elif "代码" in query or "脚本" in query or "bug" in query.lower():
        task_type = "code_help"
    else:
        task_type = "general"

    return {
        "task_type": task_type
    }

In [18]:
class AgentState(TypedDict):
    query: str
    task_type: str
    result: str
    final_answer: str

In [19]:
test_state = {
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "",
    "result": "",
    "final_answer": "",
}

In [21]:
classify_task(test_state)

{'task_type': 'experiment_analysis'}

In [22]:
test_state

{'query': '请帮我分析 coco_val_n300_g1 的幻觉风险',
 'task_type': '',
 'result': '',
 'final_answer': ''}

In [23]:
updates = classify_task(test_state)
test_state.update(updates)
test_state

{'query': '请帮我分析 coco_val_n300_g1 的幻觉风险',
 'task_type': 'experiment_analysis',
 'result': '',
 'final_answer': ''}

2.2 任务处理节点

这几个节点暂时不做真实工作，只返回 mock 结果。

In [25]:
def paper_node(state: AgentState) -> dict:
    return {
        "result": "这是论文问答任务，后续会接入论文 RAG。"
    }


def experiment_node(state: AgentState) -> dict:
    return {
        "result": "这是实验分析任务，后续会接入 CSV / JSONL 分析工具。"
    }


def dataset_node(state: AgentState) -> dict:
    return {
        "result": "这是数据集推荐任务，后续会接入数据集资料库。"
    }


def code_node(state: AgentState) -> dict:
    return {
        "result": "这是代码辅助任务，后续会接入代码解释与修改工具。"
    }


def general_node(state: AgentState) -> dict:
    return {
        "result": "这是通用科研助手任务。"
    }

In [26]:
experiment_node(test_state)

{'result': '这是实验分析任务，后续会接入 CSV / JSONL 分析工具。'}

2.3 最终回答节点 final_answer_node

这个节点负责把前面节点产生的信息整理成最终输出。

In [28]:
def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
处理结果：{state["result"]}
""".strip()

    return {
        "final_answer": final_answer
    }

In [29]:
test_state_after_node = {
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "experiment_analysis",
    "result": "这是实验分析任务，后续会接入 CSV / JSONL 分析工具。",
    "final_answer": "",
}

final_answer_node(test_state_after_node)

{'final_answer': '任务类型：experiment_analysis\n处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。'}

# 3. Conditional Edge：条件路由
你要理解

route_task 不是普通处理节点。

它的作用是：

根据当前 state["task_type"]，返回下一个节点的名字。

In [30]:
def route_task(state: AgentState) -> Literal[
    "paper_node",
    "experiment_node",
    "dataset_node",
    "code_node",
    "general_node",
]:
    task_type = state["task_type"]

    if task_type == "paper_question":
        return "paper_node"
    elif task_type == "experiment_analysis":
        return "experiment_node"
    elif task_type == "dataset_recommendation":
        return "dataset_node"
    elif task_type == "code_help":
        return "code_node"
    else:
        return "general_node"

In [31]:
route_task({
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "experiment_analysis",
    "result": "",
    "final_answer": "",
})

'experiment_node'

| 函数              | 作用       | 返回     |
| --------------- | -------- | ------ |
| `classify_task` | 修改 State | 字典     |
| `route_task`    | 决定下一步去哪  | 节点名字符串 |


# 4. Edge：固定连接

在 LangGraph 里，固定边表示：
A 节点运行完之后，一定去 B 节点

例如：

workflow.add_edge(START, "classify_task")

意思是：

流程开始后，先进入 classify_task

再比如：

workflow.add_edge("experiment_node", "final_answer")

意思是：

experiment_node 运行完后，一定进入 final_answer

# 5. 组装完整 Graph

现在我们开始真正构建 LangGraph。

In [32]:
workflow = StateGraph(AgentState)

5.1 添加节点

In [33]:
workflow.add_node("classify_task", classify_task)

workflow.add_node("paper_node", paper_node)
workflow.add_node("experiment_node", experiment_node)
workflow.add_node("dataset_node", dataset_node)
workflow.add_node("code_node", code_node)
workflow.add_node("general_node", general_node)

workflow.add_node("final_answer", final_answer_node)

先把所有函数注册成 LangGraph 里的节点。

5.2 添加起点到分类节点的固定边

In [34]:
workflow.add_edge(START, "classify_task")

START → classify_task

5.3 添加条件边

In [35]:
workflow.add_conditional_edges(
    "classify_task",
    route_task,
    {
        "paper_node": "paper_node",
        "experiment_node": "experiment_node",
        "dataset_node": "dataset_node",
        "code_node": "code_node",
        "general_node": "general_node",
    },
)

这段是今天最重要的代码之一。

它的意思是：

classify_task 运行结束后，
调用 route_task(state)，
看看 route_task 返回什么，
然后进入对应节点。

例如：

route_task(state) 返回 "experiment_node"
↓
LangGraph 进入 experiment_node

这个映射表：

{
    "paper_node": "paper_node",
    "experiment_node": "experiment_node",
    "dataset_node": "dataset_node",
    "code_node": "code_node",
    "general_node": "general_node",
}

左边是 route_task 可能返回的值，右边是真正要进入的节点名。

这里刚好一样，所以看起来有点重复。以后你也可以写成：

{
    "paper": "paper_node",
    "experiment": "experiment_node",
}

只要 route_task 返回 "paper" 或 "experiment" 就行。

# 5.4 添加任务节点到最终回答节点的固定边

In [36]:
workflow.add_edge("paper_node", "final_answer")
workflow.add_edge("experiment_node", "final_answer")
workflow.add_edge("dataset_node", "final_answer")
workflow.add_edge("code_node", "final_answer")
workflow.add_edge("general_node", "final_answer")

意思是：

无论进入哪个任务节点，最后都去 final_answer。

5.5 添加最终结束边

In [38]:
workflow.add_edge("final_answer", END)

意思是：

final_answer 运行完，流程结束。

# 6. compile：把图编译成可运行对象

In [41]:
graph = workflow.compile()

你可以理解成：

前面是在“画流程图”，compile() 是把流程图变成真正能运行的 Agent。

没有 compile()，这个图只是定义，还不能执行。

# 7. invoke：运行图

In [42]:
initial_state = {
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "",
    "result": "",
    "final_answer": "",
}

result = graph.invoke(initial_state)

result

{'query': '请帮我分析 coco_val_n300_g1 的幻觉风险',
 'task_type': 'experiment_analysis',
 'result': '这是实验分析任务，后续会接入 CSV / JSONL 分析工具。',
 'final_answer': '任务类型：experiment_analysis\n处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。'}

In [43]:
print(result["final_answer"])

任务类型：experiment_analysis
处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。


# 8. 多测几条输入

你可以定义一个小测试函数：

In [44]:
def run_agent(query: str):
    initial_state = {
        "query": query,
        "task_type": "",
        "result": "",
        "final_answer": "",
    }

    result = graph.invoke(initial_state)

    print("=" * 50)
    print("用户输入：", query)
    print(result["final_answer"])
    return result

In [45]:
run_agent("请帮我分析 coco_val_n300_g1 的幻觉风险")

用户输入： 请帮我分析 coco_val_n300_g1 的幻觉风险
任务类型：experiment_analysis
处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。


{'query': '请帮我分析 coco_val_n300_g1 的幻觉风险',
 'task_type': 'experiment_analysis',
 'result': '这是实验分析任务，后续会接入 CSV / JSONL 分析工具。',
 'final_answer': '任务类型：experiment_analysis\n处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。'}

In [46]:
run_agent("请请帮我总结一篇多模态偏见论文")

用户输入： 请请帮我总结一篇多模态偏见论文
任务类型：paper_question
处理结果：这是论文问答任务，后续会接入论文 RAG。


{'query': '请请帮我总结一篇多模态偏见论文',
 'task_type': 'paper_question',
 'result': '这是论文问答任务，后续会接入论文 RAG。',
 'final_answer': '任务类型：paper_question\n处理结果：这是论文问答任务，后续会接入论文 RAG。'}

In [47]:
run_agent("我今天应该怎么安排科研任务")

用户输入： 我今天应该怎么安排科研任务
任务类型：general
处理结果：这是通用科研助手任务。


{'query': '我今天应该怎么安排科研任务',
 'task_type': 'general',
 'result': '这是通用科研助手任务。',
 'final_answer': '任务类型：general\n处理结果：这是通用科研助手任务。'}

# 9. 观察 State 是怎么变化的

你可以在节点里加打印版，帮助理解。

比如改写 classify_task：

In [48]:
def classify_task_debug(state: AgentState) -> dict:
    print("[classify_task] 输入 state:")
    print(state)

    query = state["query"]

    if "论文" in query or "paper" in query.lower():
        task_type = "paper_question"
    elif "实验" in query or "coco" in query.lower() or "幻觉" in query:
        task_type = "experiment_analysis"
    elif "数据集" in query or "dataset" in query.lower():
        task_type = "dataset_recommendation"
    elif "代码" in query or "脚本" in query or "bug" in query.lower():
        task_type = "code_help"
    else:
        task_type = "general"

    print("[classify_task] 输出 task_type:", task_type)

    return {
        "task_type": task_type
    }

如果你想看调试信息，就改成：

def classify_task_debug(state: AgentState) -> dict:
    print("[classify_task] 输入 state:")
    print(state)

    output = classify_task(state)

    print("[classify_task] 输出:")
    print(output)

    return output




///以下为上方的完整逻辑
def classify_task_debug(state: AgentState) -> dict:
    print("[classify_task] 输入 state:")
    print(state)

    query = state["query"]

    if "论文" in query or "paper" in query.lower():
        task_type = "paper_question"
    elif "实验" in query or "coco" in query.lower() or "幻觉" in query:
        task_type = "experiment_analysis"
    elif "数据集" in query or "dataset" in query.lower():
        task_type = "dataset_recommendation"
    elif "代码" in query or "脚本" in query or "bug" in query.lower():
        task_type = "code_help"
    else:
        task_type = "general"

    print("[classify_task] 输出 task_type:", task_type)

    return {
        "task_type": task_type
    }

In [50]:
workflow = StateGraph(AgentState)

workflow.add_node("classify_task", classify_task_debug)
workflow.add_node("experiment_node", experiment_node)
workflow.add_node("general_node", general_node)
workflow.add_node("final_answer", final_answer_node)

workflow.add_edge(START, "classify_task")

workflow.add_conditional_edges(
    "classify_task",
    route_task,
    {
        "experiment_node": "experiment_node",
        "general_node": "general_node",
        "paper_node": "general_node",
        "dataset_node": "general_node",
        "code_node": "general_node",
    },
)

workflow.add_edge("experiment_node", "final_answer")
workflow.add_edge("general_node", "final_answer")
workflow.add_edge("final_answer", END)

graph = workflow.compile()

In [51]:
result = graph.invoke({
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "",
    "result": "",
    "final_answer": "",
})

print(result["final_answer"])

[classify_task] 输入 state:
{'query': '请帮我分析 coco_val_n300_g1 的幻觉风险', 'task_type': '', 'result': '', 'final_answer': ''}
[classify_task] 输出 task_type: experiment_analysis
任务类型：experiment_analysis
处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。


# 10. Day 1 最后组装成 .py 文件

当你在 notebook 里跑通后，再创建：

day1_minimal_graph.py

把下面完整代码复制进去：

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict):
    query: str
    task_type: str
    result: str
    final_answer: str


def classify_task(state: AgentState) -> dict:
    query = state["query"]

    if "论文" in query or "paper" in query.lower():
        task_type = "paper_question"
    elif "实验" in query or "coco" in query.lower() or "幻觉" in query:
        task_type = "experiment_analysis"
    elif "数据集" in query or "dataset" in query.lower():
        task_type = "dataset_recommendation"
    elif "代码" in query or "脚本" in query or "bug" in query.lower():
        task_type = "code_help"
    else:
        task_type = "general"

    return {
        "task_type": task_type
    }


def route_task(state: AgentState) -> Literal[
    "paper_node",
    "experiment_node",
    "dataset_node",
    "code_node",
    "general_node",
]:
    task_type = state["task_type"]

    if task_type == "paper_question":
        return "paper_node"
    elif task_type == "experiment_analysis":
        return "experiment_node"
    elif task_type == "dataset_recommendation":
        return "dataset_node"
    elif task_type == "code_help":
        return "code_node"
    else:
        return "general_node"


def paper_node(state: AgentState) -> dict:
    return {
        "result": "这是论文问答任务，后续会接入论文 RAG。"
    }


def experiment_node(state: AgentState) -> dict:
    return {
        "result": "这是实验分析任务，后续会接入 CSV / JSONL 分析工具。"
    }


def dataset_node(state: AgentState) -> dict:
    return {
        "result": "这是数据集推荐任务，后续会接入数据集资料库。"
    }


def code_node(state: AgentState) -> dict:
    return {
        "result": "这是代码辅助任务，后续会接入代码解释与修改工具。"
    }


def general_node(state: AgentState) -> dict:
    return {
        "result": "这是通用科研助手任务。"
    }


def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
处理结果：{state["result"]}
""".strip()

    return {
        "final_answer": final_answer
    }


def build_graph():
    workflow = StateGraph(AgentState)

    workflow.add_node("classify_task", classify_task)
    workflow.add_node("paper_node", paper_node)
    workflow.add_node("experiment_node", experiment_node)
    workflow.add_node("dataset_node", dataset_node)
    workflow.add_node("code_node", code_node)
    workflow.add_node("general_node", general_node)
    workflow.add_node("final_answer", final_answer_node)

    workflow.add_edge(START, "classify_task")

    workflow.add_conditional_edges(
        "classify_task",
        route_task,
        {
            "paper_node": "paper_node",
            "experiment_node": "experiment_node",
            "dataset_node": "dataset_node",
            "code_node": "code_node",
            "general_node": "general_node",
        },
    )

    workflow.add_edge("paper_node", "final_answer")
    workflow.add_edge("experiment_node", "final_answer")
    workflow.add_edge("dataset_node", "final_answer")
    workflow.add_edge("code_node", "final_answer")
    workflow.add_edge("general_node", "final_answer")

    workflow.add_edge("final_answer", END)

    return workflow.compile()


def run_agent(query: str):
    graph = build_graph()

    result = graph.invoke({
        "query": query,
        "task_type": "",
        "result": "",
        "final_answer": "",
    })

    return result


if __name__ == "__main__":
    while True:
        query = input("\n请输入你的科研问题，输入 q 退出：\n> ")

        if query.lower() in ["q", "quit", "exit"]:
            print("已退出。")
            break

        result = run_agent(query)

        print("\n===== Agent 输出 =====")
        print(result["final_answer"])


===== Agent 输出 =====
任务类型：general
处理结果：这是通用科研助手任务。

===== Agent 输出 =====
任务类型：paper_question
处理结果：这是论文问答任务，后续会接入论文 RAG。

===== Agent 输出 =====
任务类型：general
处理结果：这是通用科研助手任务。

===== Agent 输出 =====
任务类型：general
处理结果：这是通用科研助手任务。


运行：
python day1_minimal_graph.py

# 11. 今天你真正要想明白的东西

你不需要背代码，重点想明白这张表：

部分	在代码里对应什么	作用
State	AgentState	存储整个流程共享信息
Node	classify_task、experiment_node 等	处理 State，并返回要更新的字段
Edge	add_edge()	固定流向
Conditional Edge	add_conditional_edges() + route_task()	根据 State 决定流向
compile	workflow.compile()	把图变成可运行对象
invoke	graph.invoke(initial_state)	输入 State，执行完整图

# 12. 今天的思考题

你跑完后，认真想这 4 个问题就行：

1. classify_task 为什么返回 dict，而 route_task 返回字符串？
2. task_type 是在哪一步被写入 State 的？
3. result 是在哪一步被写入 State 的？
4. final_answer 是怎么拿到前面节点结果的？

能回答这 4 个问题，Day 1 就真的过关了。

当然，下面是这 4 个思考题的参考答案，你可以对照自己的理解。

# Day 1 思考题参考答案

## 1. `classify_task` 为什么返回 `dict`，而 `route_task` 返回字符串？

因为它们在 LangGraph 里的职责不同。

### `classify_task` 是普通节点 Node

它的作用是**更新 State**。

```python
def classify_task(state: AgentState) -> dict:
    ...
    return {
        "task_type": task_type
    }
```

它返回的是：

```python
{"task_type": "experiment_analysis"}
```

LangGraph 会把这个字典合并回 State。

也就是说，原来的 State：

```python
{
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "",
    "result": "",
    "final_answer": ""
}
```

经过 `classify_task` 后变成：

```python
{
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "experiment_analysis",
    "result": "",
    "final_answer": ""
}
```

所以：

> **普通节点返回 dict，是为了告诉 LangGraph：我要更新 State 里的哪些字段。**

---

### `route_task` 是路由函数 Router

它的作用不是更新 State，而是**决定下一步去哪个节点**。

```python
def route_task(state: AgentState) -> str:
    if state["task_type"] == "experiment_analysis":
        return "experiment_node"
```

它返回的是：

```python
"experiment_node"
```

这个字符串会被 LangGraph 用来选择下一个节点。

所以：

> **路由函数返回字符串，是为了告诉 LangGraph：下一步进入哪个节点。**

一句话总结：

```text
classify_task 返回 dict：修改 State
route_task 返回字符串：决定路线
```

---

## 2. `task_type` 是在哪一步被写入 State 的？

`task_type` 是在 `classify_task` 节点里被写入 State 的。

关键代码是：

```python
def classify_task(state: AgentState) -> dict:
    query = state["query"]

    if "论文" in query or "paper" in query.lower():
        task_type = "paper_question"
    elif "实验" in query or "coco" in query.lower() or "幻觉" in query:
        task_type = "experiment_analysis"
    elif "数据集" in query or "dataset" in query.lower():
        task_type = "dataset_recommendation"
    elif "代码" in query or "脚本" in query or "bug" in query.lower():
        task_type = "code_help"
    else:
        task_type = "general"

    return {
        "task_type": task_type
    }
```

比如用户输入：

```text
请帮我分析 coco_val_n300_g1 的幻觉风险
```

`classify_task` 会判断出：

```python
task_type = "experiment_analysis"
```

然后返回：

```python
{
    "task_type": "experiment_analysis"
}
```

LangGraph 自动合并后，State 里的 `task_type` 就被更新了。

所以答案是：

> **`task_type` 是在 `classify_task` 执行结束后，通过返回 `{"task_type": task_type}` 被写入 State 的。**

---

## 3. `result` 是在哪一步被写入 State 的？

`result` 是在具体任务节点里被写入 State 的。

比如如果任务类型是：

```python
"experiment_analysis"
```

那么路由会进入：

```python
experiment_node
```

这个节点里写入 `result`：

```python
def experiment_node(state: AgentState) -> dict:
    return {
        "result": "这是实验分析任务，后续会接入 CSV / JSONL 分析工具。"
    }
```

所以 State 会从：

```python
{
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "experiment_analysis",
    "result": "",
    "final_answer": ""
}
```

变成：

```python
{
    "query": "请帮我分析 coco_val_n300_g1 的幻觉风险",
    "task_type": "experiment_analysis",
    "result": "这是实验分析任务，后续会接入 CSV / JSONL 分析工具。",
    "final_answer": ""
}
```

不同任务会进入不同节点：

| `task_type`              | 进入节点              | 写入的 `result` |
| ------------------------ | ----------------- | ------------ |
| `paper_question`         | `paper_node`      | 论文问答任务结果     |
| `experiment_analysis`    | `experiment_node` | 实验分析任务结果     |
| `dataset_recommendation` | `dataset_node`    | 数据集推荐任务结果    |
| `code_help`              | `code_node`       | 代码辅助任务结果     |
| `general`                | `general_node`    | 通用科研助手结果     |

所以答案是：

> **`result` 是在路由后的具体任务节点中写入的，例如 `experiment_node`、`paper_node`、`dataset_node` 等。**

---

## 4. `final_answer` 是怎么拿到前面节点结果的？

`final_answer_node` 不是直接接收前一个节点单独传来的参数，而是读取完整的 State。

关键代码是：

```python
def final_answer_node(state: AgentState) -> dict:
    final_answer = f"""
任务类型：{state["task_type"]}
处理结果：{state["result"]}
""".strip()

    return {
        "final_answer": final_answer
    }
```

它从 State 里读取两个字段：

```python
state["task_type"]
state["result"]
```

这两个字段分别来自前面的节点：

```text
classify_task 写入 task_type
具体任务节点写入 result
final_answer_node 读取 task_type 和 result
```

完整流程是：

```text
初始 State
↓
classify_task 写入 task_type
↓
route_task 根据 task_type 选择节点
↓
experiment_node / paper_node / dataset_node 写入 result
↓
final_answer_node 读取 task_type 和 result，生成 final_answer
```

所以它能拿到前面结果，是因为：

> **LangGraph 会一直维护并传递同一个 State。前面节点写进去的内容，后面节点可以继续读取。**

最终它返回：

```python
{
    "final_answer": "任务类型：experiment_analysis\n处理结果：这是实验分析任务，后续会接入 CSV / JSONL 分析工具。"
}
```

然后 LangGraph 合并回 State，最终 State 里就有了完整的 `final_answer`。

---

# 最简总结版

你可以这样背：

```text
1. classify_task 返回 dict，是为了更新 State；
   route_task 返回字符串，是为了决定下一个节点。

2. task_type 是 classify_task 写入的。

3. result 是具体任务节点写入的，
   比如 experiment_node / paper_node / dataset_node。

4. final_answer_node 通过读取 State["task_type"] 和 State["result"]，
   拿到前面节点留下来的信息，然后生成 final_answer。
```

最核心的一句话：

> **LangGraph 的节点不是互相直接传参数，而是通过同一个 State 留信息、读信息、改信息。**
